# BODAQS Simple Suspension Metrics - Self-scoped

This notebook selects physical sessions directly from one configured BODAQS library and opens the shared simple suspension metrics dashboard.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import plotly.io as pio


def find_analysis_dir(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "bodaqs_analysis").is_dir():
            return candidate
        analysis = candidate / "analysis"
        if (analysis / "bodaqs_analysis").is_dir():
            return analysis
    raise RuntimeError("Could not find the BODAQS analysis package root from the current working directory.")


ANALYSIS_DIR = find_analysis_dir()
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

LIBRARIES_ROOT = Path.home() / "OneDrive" / "BODAQS-data"
LIBRARY_ID = "archie"
FALLBACK_EVENT_SCHEMA_PATH = ANALYSIS_DIR / "event schema" / "event_schema.yaml"

from bodaqs_analysis.library_api import LibraryAdapter

adapter = LibraryAdapter(LIBRARIES_ROOT)
libraries = {item["library_id"]: item for item in adapter.list_libraries()}
if LIBRARY_ID not in libraries:
    available = ", ".join(sorted(libraries)) or "none found"
    raise ValueError(f"Library {LIBRARY_ID!r} was not found. Available libraries: {available}")

LIBRARY_ROOT = Path(libraries[LIBRARY_ID]["root"])
pio.renderers.default = "notebook_connected"

FRONT_SUSPENSION_SELECTOR = {"end": "front", "domain": "wheel", "quantity": "disp", "unit": "mm"}
REAR_SUSPENSION_SELECTOR = {"end": "rear", "domain": "wheel", "quantity": "disp", "unit": "mm"}
FRONT_EVENT_SIGNAL_SELECTOR = {"end": "front", "domain": "wheel", "quantity": "disp"}
REAR_EVENT_SIGNAL_SELECTOR = {"end": "rear", "domain": "wheel", "quantity": "disp"}

SCATTER_COMPRESSION_EVENT_ID = "compressions_all>25"
SCATTER_REBOUND_EVENT_ID = "rebounds_all>25"
SCATTER_X_METRIC = "speed_mps"
SCATTER_COMPRESSION_Y_METRIC = "compression_mm"
SCATTER_REBOUND_Y_METRIC = "rebound_mm"

print(f"Analysis package root: {ANALYSIS_DIR}")
print(f"Library root: {LIBRARY_ROOT}")


Analysis package root: C:\Users\benco\dev\BODAQS\analysis
Library root: C:\Users\benco\OneDrive\BODAQS-data\archie


In [2]:
from bodaqs_analysis.dashboards import make_simple_suspension_metrics_dashboard
from bodaqs_analysis.widgets.event_schema_resolution import (
    EventSchemaResolutionError,
    resolve_event_schema_for_selection,
)
from bodaqs_analysis.widgets.session_selector import make_session_selector

sel = make_session_selector(
    artifacts_dir=LIBRARY_ROOT,
    include_aggregations=False,
    select_first_by_default=True,
    autosave_default=False,
)
display(sel["ui"])

try:
    schema_resolution = resolve_event_schema_for_selection(
        sel,
        fallback_schema_path=FALLBACK_EVENT_SCHEMA_PATH,
    )
except EventSchemaResolutionError as exc:
    raise RuntimeError("Selected sessions do not share one event schema.") from exc

for warning in schema_resolution.warnings:
    print(f"Warning: {warning}")
print(f"Schema source: {schema_resolution.source}")

dashboard = make_simple_suspension_metrics_dashboard(
    sel,
    front_displacement_selector=FRONT_SUSPENSION_SELECTOR,
    rear_displacement_selector=REAR_SUSPENSION_SELECTOR,
    front_velocity_selector=FRONT_SUSPENSION_SELECTOR,
    rear_velocity_selector=REAR_SUSPENSION_SELECTOR,
    front_event_signal_selector=FRONT_EVENT_SIGNAL_SELECTOR,
    rear_event_signal_selector=REAR_EVENT_SIGNAL_SELECTOR,
    compression_event_type=SCATTER_COMPRESSION_EVENT_ID,
    rebound_event_type=SCATTER_REBOUND_EVENT_ID,
    scatter_x_metric=SCATTER_X_METRIC,
    compression_y_metric=SCATTER_COMPRESSION_Y_METRIC,
    rebound_y_metric=SCATTER_REBOUND_Y_METRIC,
)
display(dashboard["ui"])


Schema source: frozen_artifacts
